<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [2 · One value, a domain in each language →](02_One_Value_A_Domain_In_Each_Language.ipynb)

# 1 · Values and their domains

Every value an expression denotes belongs to a **domain**: a set of values, with what may be done to them. `42` is an
integer, `2.5` a binary floating-point number, `"hi"` a string of code points. Before any expression, this tutorial
looks at domains themselves, since everything after it, from a literal's inferred domain to a C program's overflow,
reads them.

mbse-expressions invents no domain. Each is a published standard's
([PRI-012: model published standards](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-012)): ISO/IEC 11404's language-independent datatypes, IEEE 754's
floating-point formats, IEEE 1164's nine-valued logic, and Unicode's strings. Each standard is a **dialect** of its
own, and its domains are **data** in an mbse-schemas store, read by any program without the dialect's code
([CAP-097: value domains of published standards](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-097.md)). This tutorial builds such a store by hand, then
reads a domain's kind, its parameters, its schema and the edition of the standard that defines it. Each language's own
domains, built on these, are tutorial 2's.

<a id="T01.1"></a>
## 01.1 · A store that holds domains

A store starts empty (`Builders.Store()`). Each module's `install` adds its schemas: the framework's first, which says
what an expression, a dialect and a domain are, then each standard dialect's kinds of domain. Each dialect's `define`
then writes the dialect itself, a **singleton**, with the domains it owns and the edition of the standard it covers.
Each runs in a transaction of its own:

In [1]:
from mbse.Expressions import Framework
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Schemas.Framework import Builders, JSON

store = Builders.Store()
for module in (Framework, ISOIEC11404, IEEE754, IEEE1164, Unicode):
    store.transaction(module.install)
for module in (ISOIEC11404, IEEE754, IEEE1164, Unicode):
    store.transaction(module.define)
iso = store.Instances.Expressions.ISOIEC11404.Expressions  # the dialect, by its global name
print(iso)

<Expressions.ISOIEC11404.Expressions.Dialect>


The dialect is a reference object of its schema, `Expressions.ISOIEC11404.Expressions.Dialect`, and the store holds
exactly one of it, found by its global name, `Expressions.ISOIEC11404.Expressions`. `define` is idempotent: its
transaction upserts, finding before it creates, so run again it gives back the objects it wrote and changes nothing, as
the store's snapshot shows:

In [2]:
before = JSON.ToJSON(store).Snapshot()
frame = store.transaction(ISOIEC11404.define)[0]  # what it bound: the dialect, its edition, each domain
assert frame["dialect"] is iso and JSON.ToJSON(store).Snapshot() == before
print(", ".join(frame))

dialect, standard, boolean, integer, octetstring, octet, modulo_256


The toolkit's `dialects()` builds such a store with every dialect in it, each language's beside the standards', and the
rest of Part I starts from it.

<a id="T01.2"></a>
## 01.2 · A domain is data

A dialect owns its domains by name, through the framework's relation `Expressions.Framework.Defines`. `Domains.domains`
reads them, in the order the dialect's `define` wrote them; ISO/IEC 11404's are five of its datatypes. One is found by
its path, the dialect's adjacency `domains` called with the name it is owned by, and `Domains.name_of` gives the name
back:

In [3]:
from mbse.Expressions.Framework import Domains

print(", ".join(Domains.domains(iso)))
integer = iso.domains({"name": "integer"})  # a domain, by its path
assert integer is Domains.domains(iso)["integer"]
print(integer, Domains.name_of(integer))

boolean, integer, octetstring, octet, modulo(256)
<Expressions.Framework.Domain> integer


Each domain is an `Expressions.Framework.Domain`, an object like any other in the store: it is written, read, compared and
sent as JSON or YAML as any is, and a program learns what it holds by reading it, not by importing the dialect's code.
`modulo(256)` is named as ISO/IEC 11404 writes it, its modulo datatype of modulus 256.

<a id="T01.3"></a>
## 01.3 · Its kind

A domain's **kind** says what sort of set it is: an integer type, a modulo type, a floating-point format. The framework
declares a union of kinds, `Expressions.Framework.DomainKind`, and each standard dialect's install adds its own union
to it as a branch, named by the dialect's tag. `Domains.kind_of` reads a domain's: the dialect's tag, the kind's tag,
and the record holding its properties, of the kind's schema:

In [4]:
dialect, kind, record = Domains.kind_of(integer)
print(dialect, kind, record.as_schema().name)
assert record.as_schema() is store.registered("Expressions.ISOIEC11404.Expressions.OfInteger")

isoiec11404 integer Expressions.ISOIEC11404.Expressions.OfInteger


The dialect's tag comes first because two dialects may name a kind alike (C's and C++'s `nullptr_t`, tutorial 2). The
kinds are schema data too, read from the store by their registered names: the framework's union holds one branch per
standard dialect, and ISO/IEC 11404's holds the datatypes it defines:

In [5]:
DomainKind = store.registered("Expressions.Framework.DomainKind")
print([branch.name for branch in DomainKind.branches])
print([branch.name for branch in store.registered("Expressions.ISOIEC11404.Expressions.DomainKind").branches])
print({name: Domains.kind_of(domain)[1] for name, domain in Domains.domains(iso).items()})

['isoiec11404', 'ieee754', 'ieee1164', 'unicode']
['boolean', 'integer', 'modulo', 'bitstring', 'octet', 'octetstring', 'state', 'packed']
{'boolean': 'boolean', 'integer': 'integer', 'octetstring': 'octetstring', 'octet': 'octet', 'modulo(256)': 'modulo'}


ISO/IEC 11404 defines more kinds than it stores domains of: none of its five is a `bitstring`, a `state` or a `packed`
(a state value kept apart from the code that represents it); 01.5 comes back to `state`.

<a id="T01.4"></a>
## 01.4 · Its parameters, and what follows from them

A kind's record holds two sorts of property. Its **parameters** are the standard's: an integer type is `signed` or not,
of a `width` in bits or unbounded, and its `overflow` wraps, saturates or raises. Its **inferred** properties follow
from them: the `least` and `greatest` values, and whether its values are `ordered`. `Domains.properties` reads those
that are set, by name, in the kind's schema's order:

In [6]:
print(Domains.properties(integer))
wrapping = iso.domains({"name": "modulo(256)"})
print(Domains.properties(wrapping))

{'signed': True, 'overflow': 'raise', 'ordered': True}
{'modulus': 256, 'least': 0, 'greatest': 255, 'ordered': True}


ISO/IEC 11404's `integer` has no width: it is unbounded, so it has no `least` or `greatest`, and it raises on overflow,
which only a bounded integer can reach. `modulo(256)` is 11404's modulo datatype of modulus 256, the integers 0 to 255,
whose addition and multiplication wrap, as C's `uint8_t`'s do. In the standard's own text, and as its dialect's
`define` writes it in the builders' calls:

```text
modulo(256)
```

```python
_.modulo_256.kind(lambda k: k.as_value(lambda a: a.as_union(lambda u: u.isoiec11404(lambda v: v.modulo(
    lambda d: d.modulus(256).least(0).greatest(255).ordered(True))))))
```

Its modulus is its parameter; its least value, 0, its greatest, 255, and its order follow from it, and are written by
hand until constraint resolution computes them
([CAP-043: constraints resolved in any direction](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-043.md)). IEEE 754's
formats are the same: a `format` and a `rounding` attribute, then what the format fixes, its `width`, `radix`,
`precision` and largest exponent, `emax`:

In [7]:
ieee754 = store.Instances.Expressions.IEEE754.Expressions
for name, domain in Domains.domains(ieee754).items():
    print(name, Domains.kind_of(domain)[:2], Domains.properties(domain))
assert Domains.properties(ieee754.domains({"name": "binary64"}))["precision"] == 53  # Python's float, JavaScript's number

binary64 ('ieee754', 'ieee754') {'format': 'binary64', 'rounding': 'roundTiesToEven', 'width': 64, 'radix': 2, 'precision': 53, 'emax': 1023, 'ordered': True}
binary32 ('ieee754', 'ieee754') {'format': 'binary32', 'rounding': 'roundTiesToEven', 'width': 32, 'radix': 2, 'precision': 24, 'emax': 127, 'ordered': True}


<a id="T01.5"></a>
## 01.5 · Codes, strings, logic and states

Two domains can hold the same values and still differ. ISO/IEC 11404's `octet` is "`type octet = new integer range
(0..255)`": codes from 0 to 255, ordered, but not numbers, so with no arithmetic and nothing to overflow; `modulo(256)`
holds the same 256 values as numbers that wrap. Their kinds tell them apart, and their properties show it:

In [8]:
octet = iso.domains({"name": "octet"})
for domain in (octet, wrapping):
    print(Domains.name_of(domain), Domains.kind_of(domain)[1], Domains.properties(domain))

octet octet {'ordered': True}
modulo(256) modulo {'modulus': 256, 'least': 0, 'greatest': 255, 'ordered': True}


Unicode's `string` is a sequence of code points, ordered; IEEE 1164's `std_logic` is one of VHDL's nine logic states,
`U`, `X`, `0`, `1`, `Z`, `W`, `L`, `H` and `-`, unordered. Each standard dialect has a kind of its own for them, so
each is its dialect's tag twice:

In [9]:
for standard in (store.Instances.Expressions.Unicode.Expressions, store.Instances.Expressions.IEEE1164.Expressions):
    for name, domain in Domains.domains(standard).items():
        print(name, Domains.kind_of(domain)[:2], Domains.properties(domain))

string ('unicode', 'unicode') {'ordered': True}
std_logic ('ieee1164', 'ieee1164') {'ordered': False}


ISO/IEC 11404's `state` is one of its `members`, by name: an enumeration. It is a kind, but no standard here defines a
state domain, and a program's own enumerations are its own, so no dialect stores one. The one stored domain with
`members` is Excel's `error`, a kind of Excel's own: the error values a cell may hold. The toolkit's `dialects()`, a
store of every dialect, shows it, each language's domains read alike:

In [10]:
from toolkit import dialects

every = dialects()
X = every.Instances.Expressions
singletons = [X.ISOIEC11404.Expressions, X.IEEE754.Expressions, X.IEEE1164.Expressions, X.Unicode.Expressions,
              X.Patterns.Expressions, X.Python.Python3, X.TypeScript.TypeScript5, X.MATLAB.Expressions,
              X.Excel.Expressions, X.LaTeX.Expressions, X.C.Expressions, X.Cpp.Expressions, X.SystemV.AMD64,
              X.Arm.AAPCS64, X.SystemVerilog.Expressions]
stored = [domain for singleton in singletons for domain in Domains.domains(singleton).values()]
assert "state" not in {Domains.kind_of(domain)[1] for domain in stored}
print(len(stored), [Domains.name_of(domain) for domain in stored if "members" in Domains.properties(domain)])
error = X.Excel.Expressions.domains({"name": "error"})
print(Domains.kind_of(error)[:2], Domains.properties(error)["members"][:3])

135 ['error']
('excel', 'error') ('#NULL!', '#DIV/0!', '#VALUE!')


<a id="T01.6"></a>
## 01.6 · Every domain a schema

A value is written with a schema, and a domain narrows one of mbse-schemas' five scalars. So each domain has a schema
of its own, a named `Schemas.OfAny` of the scalar it narrows, which its dialect's install registers:
`Expressions.ISOIEC11404.Expressions.integer` holds `Schemas.OfInt`'s values. The framework's relation
`Expressions.Framework.HasDomain` links the two, so that each is read from the other: the domain's adjacency `schema`,
and the schema's extension adjacency `domain`, read through its builder:

In [11]:
schema = store.registered("Expressions.ISOIEC11404.Expressions.integer")
print(schema.name, [scalar.name for scalar in schema.schemas])
assert [entry.schema for entry in integer.schema] == [schema]  # the schema, from the domain
read = [entry.domain for entry in Builders.OfAny.Builder(schema, False, store).domain]  # and the domain, from the schema
print([(Domains.name_of(domain), Domains.kind_of(domain)[:2]) for domain in read])

Expressions.ISOIEC11404.Expressions.integer ['Schemas.OfInt']
[('integer', ('isoiec11404', 'integer'))]


A schema's name is a dotted identifier, so `modulo(256)`'s spells 11404's notation as one, `modulo_256`; `octet`'s is an
`OfAny` of `Schemas.OfInt` too, the same values, as 01.5 showed:

In [12]:
for domain in (wrapping, octet):
    print(Domains.name_of(domain), [entry.schema.name for entry in domain.schema])

modulo(256) ['Expressions.ISOIEC11404.Expressions.modulo_256']
octet ['Expressions.ISOIEC11404.Expressions.octet']


<a id="T01.7"></a>
## 01.7 · Standards are data

A dialect also covers the edition of its standard, through the framework's relation `Expressions.Framework.Covers`,
by its designation as its standards body writes it: an `Expressions.Framework.Standard`, holding its formal
reference's parts apart, its `bodies` (by reverse-domain name, here a joint pair), its `number` and its `revision`
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)). `Standards.standards` reads a
dialect's editions, `Standards.properties` one's parts, and `Standards.find` finds one by its designation or any name
it is known by (`Standards.tags`):

In [13]:
from mbse.Expressions.Framework import Standards

print(", ".join(Standards.standards(iso)))
edition = Standards.find(iso, "ISO/IEC 11404:2007")
print(Standards.properties(edition), Standards.tags(edition))
assert edition is Standards.standards(iso)["ISO/IEC 11404:2007"] and Standards.find(iso, "ISO/IEC 11404:1996") is None

ISO/IEC 11404:2007
{'bodies': ('org.iso', 'ch.iec'), 'number': '11404', 'revision': ('year', 2007)} ('ISO/IEC 11404:2007',)


Each standard dialect covers its own edition; Unicode's is named but not numbered:

In [14]:
for standard in (ieee754, store.Instances.Expressions.IEEE1164.Expressions, store.Instances.Expressions.Unicode.Expressions):
    for designation, edition in Standards.standards(standard).items():
        print(designation, Standards.properties(edition))

IEEE Std 754-2019 {'bodies': ('org.ieee',), 'number': '754', 'revision': ('year', 2019)}
IEEE Std 1164-1993 {'bodies': ('org.ieee',), 'number': '1164', 'revision': ('year', 1993)}
The Unicode Standard {'bodies': ('org.unicode',)}


## Why it works this way

- **Domains are the standards', not ours.** A domain names the published standard that defines it, as the standard
  names it, so that a reader who knows the standard knows the domain
  ([PRI-012: model published standards](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-012)).
- **Domains are data.** A program reads a dialect's domains from the store, as it reads any object, so it needs none
  of the dialect's code, and a store sent as JSON carries them
  ([CAP-097: value domains of published standards](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-097.md)).
- **Parameters, then what follows.** A kind holds what the standard leaves open, and what follows from it; until
  constraints are resolved in any direction, what follows is written by hand
  ([CAP-043: constraints resolved in any direction](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-043.md)).
- **A term carries no domain yet.** Inferring one for each term of an expression, from these, is a later capability
  ([CAP-232: domains inferred for terms](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-232.md)); tutorial 3 shows what inference does today.

<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [2 · One value, a domain in each language →](02_One_Value_A_Domain_In_Each_Language.ipynb)